# 13 · Type Hints and Pydantic

After this notebook you can write modern type hints and say exactly what they do (and don't do) at runtime, then use pydantic v2 to define, validate and serialise data — including generating the JSON Schema that LLM structured outputs and tool calls send, and turning a validation error into a repair prompt for the model.

**Time:** ~35 min · **Runs:** offline on CPU in < 1 min · **Needs:** [08 · Classes, OOP and dataclasses](08_classes_oop_dataclasses.ipynb), [12 · Decorators and context managers](12_decorators_and_context_managers.ipynb)

## Why this matters in interviews

- Pydantic is the backbone of LLM apps: FastAPI request models, tool schemas, structured outputs, config. "What does Pydantic do for an LLM application?" is a direct bank question.
- "How do you get reliable JSON out of a model?" — the strong answer ends with *validate with a schema, and feed the exact validation error back to the model*. You will build that loop here.
- Python rounds check that you know type hints are **not** enforced at runtime, and what `Optional`, `Literal`, `TypedDict` and `Protocol` mean.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tl20` | What does Pydantic do for an LLM application? |
| `pr07` | How do you get reliable JSON out of a model? |
| `in20` | What is structured output / constrained decoding, and how does it work? (the schema side) |
| `ag35` | What is the difference between structured output and tool calling? |
| `sd17` | Design a system to extract structured data from unstructured documents at scale. (the validation step) |

## 1. Type hints in 60 seconds

Hints document what a function expects and returns, power editor autocomplete, and let static checkers find bugs before you run anything.

| Hint | Means | Example |
|---|---|---|
| `list[int]`, `dict[str, float]`, `tuple[str, float]` | built-in generics (3.9+) | `scores: dict[str, float]` |
| `tuple[int, ...]` | a tuple of any length | `shape: tuple[int, ...]` |
| `X \| None` (3.10+) = `Optional[X]` | X or None — *not* "optional argument" | `def find(id: str) -> dict \| None` |
| `int \| str` = `Union[int, str]` | either type | `doc_id: int \| str` |
| `Literal["a", "b"]` | only these exact values | `role: Literal["system", "user"]` |
| `Final` | must not be reassigned | `MAX_TOKENS: Final = 4096` |
| `TypedDict` | a dict with known keys and value types | message dicts, JSON payloads |
| `Callable[[str], float]` | a function taking a str, returning a float | a scorer or a callback |
| `Any` | opt out of checking | escape hatch — use sparingly |
| `TypeVar` / `def f[T](...)` (3.12) | generic: "the same type in and out" | `def first[T](xs: list[T]) -> T` |
| `Protocol` | structural typing: "anything with these methods" | an `Embedder` interface |

In [ ]:
import json
import typing
from typing import Callable, Final, Literal, Optional, Protocol, TypedDict, TypeVar, runtime_checkable

Role = Literal["system", "user", "assistant"]          # a type alias
MAX_TOKENS: Final = 4096

class Message(TypedDict):
    role: Role
    content: str

Scorer = Callable[[str, str], float]                   # "a function (query, doc) -> score"
T = TypeVar("T")

def retrieve(query: str, k: int = 5, *, filters: Optional[dict[str, str]] = None) -> list[tuple[str, float]]:
    return [("refunds.md", 0.82)][:k]

def first(items: list[T]) -> T:                        # classic TypeVar spelling
    return items[0]

def last[U](items: list[U]) -> U:                      # Python 3.12 syntax for the same idea (PEP 695)
    return items[-1]

@runtime_checkable
class Embedder(Protocol):
    def embed(self, texts: list[str]) -> list[list[float]]: ...

print("hints of retrieve():", typing.get_type_hints(retrieve))
print("first/last:", first(["a", "b"]), last([1, 2, 3]))

## 2. Hints are **not** enforced at runtime

**Predict:** what does `add_tokens("4", "2")` return, given the hints say `int`?

In [ ]:
def add_tokens(a: int, b: int) -> int:
    return a + b

print("add_tokens('4', '2') ->", repr(add_tokens("4", "2")))       # no error: hints are just metadata
count: int = "not a number"                                         # also no error
msg: Message = {"role": "robot", "content": 42}                    # a TypedDict is a plain dict at runtime
print("count:", count, "| msg:", msg, "| type(msg):", type(msg).__name__)

class HashEmbedder:
    def embed(self, texts):
        return [[float(hash(t) % 100)] for t in texts]

class Summariser:
    def summarise(self, text):
        return text[:50]

print("HashEmbedder is an Embedder?", isinstance(HashEmbedder(), Embedder))
print("Summariser is an Embedder?  ", isinstance(Summariser(), Embedder))
assert add_tokens("4", "2") == "42"

Python ignored every hint. They are checked by **static type checkers** — `mypy`, `pyright` (the engine inside VS Code's Pylance) — which read your code without running it, typically in the editor and in CI. They are not installed in this environment; on the cell above, mypy would report errors like these (plus one for each bad value in the `TypedDict`):

```text
nb.py:4: error: Argument 1 to "add_tokens" has incompatible type "str"; expected "int"  [arg-type]
nb.py:5: error: Incompatible types in assignment (expression has type "str", variable has type "int")  [assignment]
```

`@runtime_checkable` lets `isinstance` check a `Protocol`, but only that the **method names exist** — not their signatures. When you need real runtime checking of data coming from outside (an HTTP request, a config file, **a model's output**), you need a validation library. That is pydantic.

## 3. Pydantic v2: `BaseModel`

**In plain English:** you describe the shape of the data once, as a class with type hints. Pydantic then checks every incoming value against that description, converts what it safely can (the string `"42"` into the int `42`), and raises one clear error listing everything that is wrong. The core is written in Rust, so it is fast enough to validate every request.

In [ ]:
from pydantic import BaseModel, ConfigDict, Field, TypeAdapter, ValidationError, field_validator, model_validator

class Ticket(BaseModel):
    id: int
    title: str
    priority: Literal["low", "medium", "high"] = "medium"
    tags: list[str] = []                    # safe in pydantic: every instance gets its own copy

t = Ticket(id="42", title="Refund not received")      # "42" is converted to 42
print(t)
print("t.id + 1 =", t.id + 1, "| fields:", list(Ticket.model_fields))

a, b = Ticket(id=1, title="a"), Ticket(id=2, title="b")
a.tags.append("billing")
print("mutable default shared?", b.tags)               # [] - unlike a plain function default (notebook 16)

try:
    Ticket(id="seven", title=None, priority="urgent")
except ValidationError as e:
    print(f"\n{e.error_count()} errors, all reported at once:")
    for err in e.errors(include_url=False):
        print(f"  {err['loc']}: {err['msg']}  [type={err['type']}, input={err['input']!r}]")
assert t.id == 42 and b.tags == []

Every error has a `loc` (where), a `msg` (human-readable), a `type` (a stable machine-readable code) and the `input` it got. `str(e)` prints the same with a documentation link; `e.json()` gives it as JSON. Keep that structure in mind — it is exactly what we will feed back to a model in section 11.

## 4. `Field`: constraints, descriptions, aliases, defaults

- **Constraints**: `ge`/`gt`/`le`/`lt` for numbers, `min_length`/`max_length`, `pattern` for strings and lists.
- **`description`**: documentation that ends up in the JSON Schema — and therefore in the prompt the model sees.
- **`alias`**: accept `topK` from a JavaScript client while your Python attribute stays `top_k`.
- **`default_factory`**: build a fresh default per instance (`default_factory=list`).

In [ ]:
class SearchRequest(BaseModel):
    model_config = ConfigDict(validate_by_name=True, validate_by_alias=True)   # accept top_k and topK

    query: str = Field(min_length=1, max_length=500, description="What the user is looking for.")
    top_k: int = Field(default=5, ge=1, le=50, alias="topK", description="How many chunks to return.")
    min_score: float = Field(default=0.0, ge=0.0, le=1.0)
    sources: list[str] = Field(default_factory=list, description="Restrict the search to these files.")

print(SearchRequest(query="refund policy", topK=8))
print(SearchRequest(query="refund policy", top_k=8).model_dump(by_alias=True))
try:
    SearchRequest(query="", topK=500)
except ValidationError as e:
    print([f"{err['loc'][0]}: {err['msg']}" for err in e.errors()])

(`populate_by_name=True` is the older spelling of "also accept the field name"; pydantic 2.11+ splits it into `validate_by_name` / `validate_by_alias`.)

## 5. Coercion (lax mode) vs strict mode

By default pydantic is **lax**: it converts values when the conversion is unambiguous and loses nothing. That is useful with LLM output, where a model often writes `"0.87"` or `"5"` as strings. **Strict** mode refuses any conversion — right for internal APIs where a string where an int belongs signals a bug.

| Input for `n: int` | lax (default) | strict |
|---|---|---|
| `42` | 42 | 42 |
| `"42"` | 42 | error |
| `3.0` | 3 | error |
| `3.5` | error (would lose data) | error |
| `"yes"` for a `bool` | True | error |

In [ ]:
class Reading(BaseModel):
    n: int
    x: float
    flag: bool

for data in [{"n": "42", "x": "3.5", "flag": "yes"}, {"n": 3.0, "x": 1, "flag": 1}, {"n": 3.5, "x": 1, "flag": "maybe"}]:
    try:
        print(f"lax    {data} -> {Reading(**data)}")
    except ValidationError as e:
        print(f"lax    {data} -> errors {[(err['loc'][0], err['type']) for err in e.errors()]}")
try:
    Reading.model_validate({"n": "42", "x": 1.0, "flag": True}, strict=True)
except ValidationError as e:
    print(f"strict {{'n': '42', ...}} -> errors {[(err['loc'][0], err['type']) for err in e.errors()]}")

You can also make one field strict (`Field(strict=True)`) or a whole model (`model_config = ConfigDict(strict=True)`).

## 6. Validators and nested models

- `@field_validator("x")` — custom logic for one field. `mode="before"` runs on the raw input (normalise `" usd "` to `"USD"`); the default `mode="after"` runs on the already-typed value.
- `@model_validator(mode="after")` — rules that involve **several fields**: an invoice total must equal the sum of its line items. This is how you encode business rules in the type itself, which is far stronger than asking the model nicely in the prompt.
- Models nest: `items: list[LineItem]` validates every element, and errors point at the exact element (`items[0].quantity`).

This invoice model is the one we will extract from text in the practical section.

In [ ]:
class LineItem(BaseModel):
    description: str = Field(min_length=1)
    quantity: int = Field(gt=0)
    unit_price: float = Field(ge=0, description="Price of one unit, before tax.")

class Invoice(BaseModel):
    model_config = ConfigDict(extra="forbid")                  # unknown keys are errors, not silently kept

    invoice_id: str = Field(pattern=r"^INV-\d{4}$", description="Invoice number, e.g. INV-0042.")
    currency: Literal["USD", "EUR", "INR"]
    items: list[LineItem] = Field(min_length=1)
    total: float = Field(description="Grand total. Must equal the sum of quantity * unit_price.")

    @field_validator("currency", mode="before")
    @classmethod
    def normalise_currency(cls, v):
        return v.strip().upper() if isinstance(v, str) else v  # forgive " usd " - not worth a retry

    @model_validator(mode="after")
    def total_matches_items(self):
        expected = round(sum(i.quantity * i.unit_price for i in self.items), 2)
        if abs(expected - self.total) > 0.01:
            raise ValueError(f"total {self.total} does not equal the sum of the line items ({expected})")
        return self

good = Invoice.model_validate({"invoice_id": "INV-0042", "currency": " usd ",
                               "items": [{"description": "API credits", "quantity": 2, "unit_price": 10}], "total": 20})
print(good)
for bad in [{"invoice_id": "INV-0042", "currency": "USD", "items": [{"description": "API credits", "quantity": 2, "unit_price": 10}], "total": 25},
            {"invoice_id": "INV-0042", "currency": "USD", "items": [{"description": "", "quantity": 0, "unit_price": 10}], "total": 0}]:
    try:
        Invoice.model_validate(bad)
    except ValidationError as e:
        print([f"{err['loc']}: {err['msg']}" for err in e.errors()])
assert good.currency == "USD"

Notice the order: field-level checks run first; the model validator only runs once every field is valid (its error has an empty `loc`, meaning "the object as a whole").

## 7. Serialising and parsing

| Direction | Method | Notes |
|---|---|---|
| model → dict | `model_dump()` | `mode="json"` makes everything JSON-safe (datetimes → strings); `exclude_none=True`, `include=`, `exclude=`, `by_alias=True` |
| model → JSON string | `model_dump_json()` | fast; `indent=2` for humans |
| dict → model | `Model.model_validate(data)` | full validation |
| JSON string → model | `Model.model_validate_json(text)` | parses and validates in one step, in Rust — prefer it for raw model output |
| copy with changes | `m.model_copy(update={...})` | note: the update is **not** validated |

In [ ]:
from datetime import datetime, timezone

class ChatLog(BaseModel):
    request_id: str
    model: str
    created: datetime
    latency_ms: int | None = None

log = ChatLog(request_id="req-1", model="gpt-4.1-mini", created=datetime(2026, 9, 29, 9, 30, tzinfo=timezone.utc))
print("model_dump()         :", log.model_dump())
print("model_dump(mode=json):", log.model_dump(mode="json", exclude_none=True))
print("model_dump_json()    :", log.model_dump_json())
back = ChatLog.model_validate_json(log.model_dump_json())
print("round trip equal?    :", back == log)
print("model_copy(update=)  :", log.model_copy(update={"latency_ms": 812}).latency_ms)
assert back == log and isinstance(log.model_dump(mode="json")["created"], str)

## 8. `model_json_schema()` — exactly what structured outputs send

This is the bridge to LLMs (in20, tl20). `Invoice.model_json_schema()` produces a standard JSON Schema: types, required fields, enums from `Literal`, the `pattern`, nested models under `$defs`, `additionalProperties: false` from `extra="forbid"`, and every `description` you wrote — which the model reads as instructions. With **structured outputs**, the provider enforces that schema while decoding (tokens that would break it are masked out), so the reply is valid JSON of the right shape by construction. The *values* can still be wrong — which is why the model validator matters.

In [ ]:
schema = Invoice.model_json_schema()
print("top-level keys :", list(schema))
print("required       :", schema["required"])
print("currency       :", schema["properties"]["currency"])
print("invoice_id     :", schema["properties"]["invoice_id"])
print("items          :", schema["properties"]["items"])
print("LineItem fields:", list(schema["$defs"]["LineItem"]["properties"]))
assert schema["additionalProperties"] is False and schema["properties"]["currency"]["enum"] == ["USD", "EUR", "INR"]

The same model can become a **tool** definition (ag35: structured output constrains the final answer; tool calling constrains an action your code will run — mechanically the same schema). The `openai` SDK's `pydantic_function_tool` converts a model into a strict function-calling schema: it adds `"strict": true` and `additionalProperties: false` on every nested object too, as strict mode requires.

In [ ]:
import openai

tool_def = openai.pydantic_function_tool(Invoice, name="record_invoice", description="Save an extracted invoice.")
print(json.dumps({k: v for k, v in tool_def["function"].items() if k != "parameters"}, indent=2))
nested_closed = tool_def["function"]["parameters"]["$defs"]["LineItem"].get("additionalProperties") is False
print("nested LineItem also has additionalProperties: false ->", nested_closed)
assert tool_def["function"]["strict"] is True and nested_closed

With a live model, the SDK does the whole round trip for you (see [structured outputs with Pydantic](../08_llm_apps/03_structured_outputs_with_pydantic.ipynb)):

```python
completion = client.chat.completions.parse(model=MODEL, messages=messages, response_format=Invoice)
invoice = completion.choices[0].message.parsed      # an Invoice instance (None if the model refused)
```

The SDK validates the reply with **your** model, so a failing validator — like `total_matches_items` — still raises `ValidationError`, and the repair loop below is still needed.

## 9. `Enum` and `Literal` fields

Both restrict a field to fixed values and both become `enum` in the schema. `Literal` is lighter (plain strings); an `Enum` gives you a named type you can import, iterate and attach methods to. A `StrEnum` member *is* a string, so it compares equal to `"refund"` and serialises cleanly.

In [ ]:
from enum import StrEnum

class Intent(StrEnum):
    REFUND = "refund"
    SHIPPING = "shipping"
    OTHER = "other"

class Classification(BaseModel):
    intent: Intent
    confidence: float = Field(ge=0, le=1)

c = Classification.model_validate_json('{"intent": "refund", "confidence": "0.91"}')
print(c, "| equals the string:", c.intent == "refund", "| dump:", c.model_dump_json())
print("schema:", Classification.model_json_schema()["$defs"]["Intent"])
try:
    Classification(intent="complaint", confidence=0.5)
except ValidationError as e:
    print("error:", e.errors()[0]["msg"])

## 10. `TypeAdapter` — validate things that are not models

Need to validate a `list[int]`, a `dict[str, float]`, a `TypedDict` or a dataclass without writing a `BaseModel`? Wrap the type in a `TypeAdapter`. It has the same `validate_python`, `validate_json`, `dump_json` and `json_schema` methods.

In [ ]:
class Hit(TypedDict):
    doc_id: str
    score: float

hits_adapter = TypeAdapter(list[Hit])
hits = hits_adapter.validate_json('[{"doc_id": "refunds.md", "score": "0.82"}, {"doc_id": "faq.md", "score": 0.4}]')
print("validated hits:", hits)
print("list[int]     :", TypeAdapter(list[int]).validate_python(["1", 2, 3.0]))
print("schema        :", TypeAdapter(dict[str, float]).json_schema())
try:
    hits_adapter.validate_python([{"doc_id": "x"}])
except ValidationError as e:
    print("error at", e.errors()[0]["loc"], "->", e.errors()[0]["msg"])
assert hits[0]["score"] == 0.82

## 11. dataclass vs TypedDict vs pydantic — which one when?

| | `@dataclass` | `TypedDict` | pydantic `BaseModel` |
|---|---|---|---|
| What it is at runtime | a class | a plain `dict` | a class with a validator |
| Validates types at runtime | no | no | **yes** (with coercion) |
| JSON Schema | via `TypeAdapter` | via `TypeAdapter` | `model_json_schema()` |
| Parse / dump JSON | manual | `json` module | built in |
| Cost to create | cheapest object | cheapest overall | highest (it validates) |
| Use it for | internal data you create yourself | typing existing dicts / JSON payloads | **trust boundaries**: requests, config, tool arguments, model output |

The cost difference is real but small — microseconds — so validate at the boundaries, not in tight inner loops. The chart also includes `model_construct()`, which builds a model **without** validating (for data you already trust). In v2 it runs in Python while validation runs in Rust, so it is not the speed-up people expect — measure before reaching for it.

In [ ]:
import timeit
from dataclasses import dataclass

import matplotlib.pyplot as plt

BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e1e0d9", "axes.axisbelow": True})

@dataclass
class HitDC:
    doc_id: str
    score: float

class HitPD(BaseModel):
    doc_id: str
    score: float

raw = {"doc_id": "refunds.md", "score": 0.82}
contenders = {
    "dict literal": lambda: {"doc_id": "refunds.md", "score": 0.82},
    "TypedDict": lambda: Hit(doc_id="refunds.md", score=0.82),
    "dataclass": lambda: HitDC("refunds.md", 0.82),
    "pydantic model_construct\n(no validation)": lambda: HitPD.model_construct(**raw),
    "pydantic model_validate": lambda: HitPD.model_validate(raw),
}
n = 20_000
cost_us = {name: min(timeit.repeat(fn, number=n, repeat=5)) / n * 1e6 for name, fn in contenders.items()}
for name, us in cost_us.items():
    print(f"{name.replace(chr(10), ' '):<42} {us:6.2f} µs per object")

fig, ax = plt.subplots(figsize=(8.5, 3.4))
colors = [BLUE, BLUE, BLUE, AQUA, ORANGE]
bars = ax.barh(list(cost_us), list(cost_us.values()), color=colors, height=0.6)
ax.bar_label(bars, fmt="%.2f µs", padding=3, fontsize=9)
ax.invert_yaxis(); ax.grid(axis="y", visible=False)
ax.set_xlabel("microseconds to create one object (lower is faster)")
ax.set_title("Validation costs microseconds: do it at trust boundaries")
plt.show()
assert cost_us["pydantic model_validate"] > cost_us["dataclass"]

## 12. Practical: validate an LLM extraction and build a repair prompt

The loop from `pr07` and `tl20`:

```
prompt + schema ──► model ──► reply ──► strip fences ──► Invoice.model_validate_json
                      ▲                                        │
                      │                          valid ────────┴──► done
                      └── repair prompt: the exact errors ◄── ValidationError (max N attempts)
```

A generic "that was wrong, try again" rarely works; the **specific** validator messages ("items[0].quantity: Input should be a valid integer; you sent 'two'") almost always get fixed on the next attempt. Cap the attempts — an unbounded re-ask loop is a money pump.

To keep this deterministic, a scripted fake model plays the part of the LLM, making the mistakes real models make: a markdown fence, a wrong id format, a number written as a word, an extra key — and then an arithmetic error that only the business rule catches.

In [ ]:
import re

FENCE = re.compile(r"^\s*```(?:json)?\s*(.*?)\s*```\s*$", re.DOTALL)

def loc_to_path(loc):
    path = ""
    for part in loc:
        path += f"[{part}]" if isinstance(part, int) else (f".{part}" if path else str(part))
    return path or "(whole object)"

def repair_prompt(err):
    lines = []
    for e in err.errors(include_url=False):
        sent = f"; you sent {e['input']!r}" if e["loc"] else ""
        msg = e["msg"].removeprefix("Value error, ")
        lines.append(f"- {loc_to_path(e['loc'])}: {msg}{sent}")
    return ("Your JSON did not match the required schema. Fix these problems and reply with only the corrected JSON:\n"
            + "\n".join(lines))

class ScriptedLLM:
    """Stands in for client.chat.completions.create: returns canned replies in order."""
    def __init__(self, replies):
        self.replies, self.calls = list(replies), 0

    def complete(self, messages):
        self.calls += 1
        return self.replies.pop(0)

In [ ]:
def extract_invoice(text, llm, max_attempts=3):
    messages = [{"role": "system", "content": "Extract the invoice as JSON matching this schema:\n"
                                              + json.dumps(Invoice.model_json_schema())},
                {"role": "user", "content": text}]
    for attempt in range(1, max_attempts + 1):
        reply = llm.complete(messages)
        cleaned = FENCE.sub(r"\1", reply)                       # cheap, safe fix: no retry needed for fences
        try:
            return Invoice.model_validate_json(cleaned), attempt, messages
        except ValidationError as err:
            messages += [{"role": "assistant", "content": reply}, {"role": "user", "content": repair_prompt(err)}]
    raise RuntimeError(f"no valid invoice after {max_attempts} attempts")   # a named failure, not a bad object

llm = ScriptedLLM([
    '```json\n{"invoice_id": "42", "currency": "usd", "items": [{"description": "API credits", '
    '"quantity": "two", "unit_price": 10}], "total": 20, "notes": "thanks!"}\n```',
    '{"invoice_id": "INV-0042", "currency": "USD", "items": [{"description": "API credits", '
    '"quantity": 2, "unit_price": 10}], "total": 25}',
    '{"invoice_id": "INV-0042", "currency": "USD", "items": [{"description": "API credits", '
    '"quantity": 2, "unit_price": 10}], "total": 20}',
])
invoice, attempts, transcript = extract_invoice("Invoice INV-0042: 2 x API credits at $10 each. Total $20.", llm)

for i, m in enumerate(transcript[2:], start=1):
    if m["role"] == "user":
        print(f"--- repair prompt after attempt {i // 2} ---\n{m['content']}\n")
print(f"valid after {attempts} attempts:", invoice)
assert attempts == 3 and llm.calls == 3 and invoice.total == 20.0
assert "items[0].quantity" in transcript[3]["content"] and "does not equal" in transcript[5]["content"]

Read the two repair prompts as the model would: every problem is named with its location, the rule, and the value it sent. Attempt 1's errors are schema errors (a provider's structured-output mode would have prevented them); attempt 2's is a **business rule** that no schema can express — only your validator catches it, and only a specific message gets it fixed.

Two production notes: log how many attempts each extraction needed (a rising repair rate is an early warning of prompt or model drift), and remember that *schema-valid is not correct* — a well-formed invoice with an id that does not exist in your system passes every check above. Exercise 3 adds that check.

## Try it yourself

**1.** Write a `DateRange` model with `start: date` and `end: date` and a model validator that rejects `start > end`. Check that ISO strings like `"2026-09-01"` are accepted.

In [ ]:
# Solution — try it yourself first, then run this
from datetime import date

class DateRange(BaseModel):
    start: date
    end: date

    @model_validator(mode="after")
    def ordered(self):
        if self.start > self.end:
            raise ValueError(f"start {self.start} is after end {self.end}")
        return self

print(DateRange(start="2026-09-01", end="2026-09-30"))
try:
    DateRange(start="2026-10-01", end="2026-09-30")
except ValidationError as e:
    print("rejected:", e.errors()[0]["msg"])
assert DateRange(start="2026-09-01", end="2026-09-30").start == date(2026, 9, 1)

**2.** Retrieval results arrive as JSON: `[{"doc_id": "a.md", "score": "0.4"}, {"doc_id": "b.md", "score": 0.9}]`. Validate them with a `TypeAdapter` over a small `BaseModel`, and return them sorted by score, highest first.

In [ ]:
# Solution — try it yourself first, then run this
class ScoredDoc(BaseModel):
    doc_id: str
    score: float = Field(ge=0, le=1)

payload = '[{"doc_id": "a.md", "score": "0.4"}, {"doc_id": "b.md", "score": 0.9}]'
ranked = sorted(TypeAdapter(list[ScoredDoc]).validate_json(payload), key=lambda d: d.score, reverse=True)
print(ranked)
assert [d.doc_id for d in ranked] == ["b.md", "a.md"]

**3.** Schema-valid is not correct. Use pydantic's **validation context** to check that an extracted `invoice_id` actually exists in your system: a validator can read `info.context`, which you pass as `model_validate(..., context={...})`.

In [ ]:
# Solution — try it yourself first, then run this
from pydantic import ValidationInfo

class KnownInvoice(BaseModel):
    invoice_id: str = Field(pattern=r"^INV-\d{4}$")

    @field_validator("invoice_id")
    @classmethod
    def must_exist(cls, v, info: ValidationInfo):
        known = (info.context or {}).get("known_ids", set())
        if v not in known:
            raise ValueError(f"{v} is not an invoice in our system")
        return v

ctx = {"known_ids": {"INV-0042", "INV-0043"}}
print(KnownInvoice.model_validate({"invoice_id": "INV-0042"}, context=ctx))
rejected = False
try:
    KnownInvoice.model_validate({"invoice_id": "INV-9999"}, context=ctx)   # well-formed, but hallucinated
except ValidationError as e:
    rejected = True
    print("rejected:", e.errors()[0]["msg"])
assert rejected

## Say it in an interview

- **Type hints (30 seconds):** "Hints are metadata — Python does not enforce them at runtime. They power editors and static checkers like mypy and pyright in CI. `X | None` means the value can be None, not that the argument is optional. `Protocol` is structural typing; `TypedDict` types a plain dict."
- **What pydantic does (tl20):** "One definition, three jobs: validate inbound data (FastAPI requests, tool arguments, model output), generate the JSON Schema that tool definitions and structured outputs send, and serialise back out. Errors are structured — location, message, type, input — so a failed model reply becomes a precise repair prompt. Validators encode business rules in the type."
- **Reliable JSON (pr07 / in20):** "Best: provider structured outputs — constrained decoding makes the shape valid by construction. Then validate the values with the same pydantic model, and on failure send the exact errors back, capped at two or three attempts. Schema-valid is not correct: check values against your own data."
- **Structured output vs tool calling (ag35):** "Same mechanism — a JSON Schema constraining the output. Structured output is the final answer; a tool call is an intermediate action your code executes and feeds back."
- **Numbers:** validation costs microseconds per object — negligible at a boundary, noticeable in a hot loop of millions.
- **Traps:** lax coercion (`"42"` becomes 42) — use strict where a type mismatch means a bug; `model_copy(update=...)` skips validation; `extra` defaults to *ignore*, so unknown keys vanish silently unless you set `extra="forbid"`; mixing pydantic v1 and v2 APIs (`.dict()`/`parse_obj` are v1; v2 is `model_dump`/`model_validate`); an unbounded re-ask loop.

## Next

- [14 · async/await basics](14_async_await_basics.ipynb)
- [15 · Testing and debugging](15_testing_and_debugging.ipynb) — testing the repair loop with mocks
- LLM apps: [structured outputs with Pydantic](../08_llm_apps/03_structured_outputs_with_pydantic.ipynb) · [function calling](../08_llm_apps/04_function_calling.ipynb) · [LangChain structured output and tools](../13_langchain/03_structured_output_and_tools.ipynb)
- Serving: [FastAPI serving and streaming](../16_production/04_fastapi_serving_and_streaming.ipynb) — pydantic models as request and response types
- Theory: [python_basics course](../../python_basics/index.html) · [genai_flow course](../../genai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)